# 🎮 Halo CE Xbox (Build 2276) — MSVC 7.1 Byte-Matching & Verification Pipeline

> **Repository:** [nickarcade/halo](https://github.com/nickarcade/halo)  
> **Target Branch:** `mainrevised2`  
> **Target Binary:** Pristine Halo Xbox Debug Build 2276 (`cachebeta.xbe`, MD5: `c7869590a1c64ad034e49a5ee0c02465`)  
> **Compiler Toolchain:** Microsoft Visual C++ Toolkit 2003 (`CL.Exe` v13.10.3077 / MSVC 7.1) via Wine32  
> **Byte-Matching Suite:** `tools/bytematch/bytematch.py` (AST mutation search loop), `tools/bytematch/pal_campaign.py`, and `tools/verify/raw_xbe_structural.py`

---
### 📋 Pipeline Workflow Overview
1. **Mount Google Drive & Stage Assets** (`VCToolkitSetup.exe`, `cachebeta.xbe`)
2. **Clone / Synchronize Workspace** from branch `mainrevised2`
3. **Install Dependencies** (Wine 32-bit, LLVM objdump, `libclang`, `pefile`, `pyxbe`)
4. **Install Headless MSVC 7.1 & Configure Transparent Shims** (`/mnt/c/.../CL.Exe`, `wslpath -> winepath`)
5. **Verify Toolchain & Generate Headers** (`decl.h`)
6. **Execute Canonical VC71 Verification & Score Sweep** (`vc71_verify.py`)
7. **Run AST-Guided Byte-Matching Loop** (`bytematch.py queue`, `search`, `rules`, and patch generation)


In [1]:
# @title 🛠️ Step 1: Environment Setup & Toolchain Installation
import os
import sys
import shutil
import subprocess
import hashlib
import time
import urllib.request
print("=" * 80)
print("  HALO CE XBOX (BUILD 2276) - MSVC 7.1 & BYTEMATCH ENVIRONMENT SETUP")
print("  Branch: mainrevised2 | Target: cachebeta.xbe")
print("=" * 80)
# --- 1. OPTIONAL GOOGLE DRIVE MOUNT & ASSET STAGING ---
print("\n>>> [1/6] Staging Assets from Google Drive / Web (if available)...")
drive_paths = [
    "/content/drive/MyDrive/MSVC 7.1",
    "/content/drive/MyDrive/Colab Notebooks",
    "/content/drive/MyDrive"
]
assets = [
    ("VCToolkitSetup.exe", "/content/VCToolkitSetup.exe"),
    ("cachebeta.xbe", "/content/cachebeta.xbe"),
]
for filename, dest in assets:
    if not os.path.exists(dest):
        for dp in drive_paths:
            candidate = os.path.join(dp, filename)
            if os.path.exists(candidate):
                print(f"  Staging {filename} from {dp}...")
                shutil.copy2(candidate, dest)
                break
# If VCToolkitSetup.exe is still missing, auto-download from archive.org
if not os.path.exists("/content/VCToolkitSetup.exe"):
    vc_urls = [
        "https://archive.org/download/microsoft-visual-c-toolkit-2003/VCToolkitSetup.exe",
        "https://archive.org/download/VCToolkitSetup/VCToolkitSetup.exe",
        "https://web.archive.org/web/20070208115507/http://download.microsoft.com/download/d/3/8/d3844074-8b01-447a-85d9-482a52ee2407/VCToolkitSetup.exe"
    ]
    for u in vc_urls:
        try:
            print(f"  Fetching VCToolkitSetup.exe from {u}...")
            req = urllib.request.Request(u, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=30) as resp:
                if resp.status == 200:
                    data = resp.read()
                    if len(data) > 1000000:
                        with open("/content/VCToolkitSetup.exe", "wb") as f:
                            f.write(data)
                        print(f"  ✓ Downloaded VCToolkitSetup.exe ({len(data):,} bytes)")
                        break
        except Exception as e:
            print(f"  Attempt failed for {u}: {e}")
for filename, dest in assets:
    if os.path.exists(dest):
        print(f"  ✓ Asset ready: {dest} ({os.path.getsize(dest):,} bytes)")
    else:
        print(f"  ⚠ Asset missing: {filename}. Please upload to /content/{filename} if needed.")
# --- 2. CLONE / UPDATE REPOSITORY (BRANCH: mainrevised2) ---
print("\n>>> [2/6] Synchronizing repository (branch: mainrevised2)...")
os.chdir("/content")
halo_dir = "/content/halo"
repo_url = "https://github.com/nickarcade/halo.git"
if not os.path.exists(os.path.join(halo_dir, ".git")):
    if os.path.exists(halo_dir):
        shutil.rmtree(halo_dir)
    print(f"  Cloning {repo_url} on branch 'mainrevised2'...")
    subprocess.run(["git", "clone", "-b", "mainrevised2", repo_url, halo_dir], check=True)
else:
    print("  Repository already present. Pulling latest mainrevised2...")
    subprocess.run(["git", "-C", halo_dir, "fetch", "origin", "mainrevised2"], check=True)
    subprocess.run(["git", "-C", halo_dir, "checkout", "mainrevised2"], check=True)
    subprocess.run(["git", "-C", halo_dir, "pull", "origin", "mainrevised2"], check=True)
# Stage pristine cachebeta.xbe into halo-patched/
target_xbe_dir = os.path.join(halo_dir, "halo-patched")
os.makedirs(target_xbe_dir, exist_ok=True)
dest_xbe = os.path.join(target_xbe_dir, "cachebeta.xbe")
if os.path.exists("/content/cachebeta.xbe"):
    if not os.path.exists(dest_xbe) or os.path.getsize(dest_xbe) != os.path.getsize("/content/cachebeta.xbe"):
        shutil.copy2("/content/cachebeta.xbe", dest_xbe)
if os.path.exists(dest_xbe):
    with open(dest_xbe, "rb") as f:
        md5 = hashlib.md5(f.read()).hexdigest()
    expected = "c7869590a1c64ad034e49a5ee0c02465"
    status_tag = "[MATCH]" if md5 == expected else "[MISMATCH!]"
    print(f"  ✓ cachebeta.xbe verified in halo-patched/ (MD5: {md5}) {status_tag}")
else:
    print("  ⚠ cachebeta.xbe not yet in halo-patched/. Ensure it is staged for verification.")
# --- 3. SYSTEM & PYTHON DEPENDENCIES ---
print("\n>>> [3/6] Installing Linux packages (Wine32, LLVM, libclang, Python analysis tools)...")
subprocess.run(["sudo", "dpkg", "--add-architecture", "i386"], check=True)
subprocess.run(["sudo", "apt-get", "update", "-qq"], check=True)
subprocess.run(["sudo", "apt-get", "install", "-y", "-qq", "wine32:i386", "wine", "llvm", "xvfb"], check=True)
# Install Python packages including libclang (vital for bytematch.py AST search)
subprocess.run([
    "pip", "install", "-q",
    "pefile~=2023.2.7", "pyxbe~=1.0.2", "pytest", "capstone", "libclang",
    "--break-system-packages"
], check=True)
print("  ✓ Toolchain & Python packages installed successfully.")
# --- 4. HEADLESS MSVC COMPILER INSTALLATION ---
print("\n>>> [4/6] Installing Microsoft Visual C++ Toolkit 2003 in Wine...")
wine_vc_dir = os.path.expanduser("~/.wine/drive_c/Program Files/Microsoft Visual C++ Toolkit 2003")
cl_path = os.path.join(wine_vc_dir, "bin", "cl.exe")
if not os.path.exists(cl_path):
    if not os.path.exists("/content/VCToolkitSetup.exe"):
        raise FileNotFoundError("Missing /content/VCToolkitSetup.exe! Please upload it to /content or Google Drive.")
    env = os.environ.copy()
    env["WINEARCH"] = "win32"
    env["WINEDEBUG"] = "-all"
    subprocess.run(["wineserver", "-k"], env=env)
    print("  Running silent installer via xvfb...")
    subprocess.run(
        ["xvfb-run", "--auto-servernum", "wine", "/content/VCToolkitSetup.exe", "/s", "/v/qn"],
        env=env,
        check=True,
        timeout=120
    )
print(f"  ✓ MSVC Toolkit compiler verified: {cl_path}")
# --- 5. CONFIGURE TRANSPARENT WINE SHIMS FOR REPO SCRIPTS ---
print("\n>>> [5/6] Configuring transparent WSL/Wine shims for vc71_verify.py and bytematch.py...")
# 1) wslpath shim -> winepath
with open("/usr/local/bin/wslpath", "w") as f:
    f.write('#!/bin/bash\nif [ "$1" = "-w" ]; then winepath -w "$2"; elif [ "$1" = "-u" ]; then winepath -u "$2"; else winepath "$@"; fi\n')
os.chmod("/usr/local/bin/wslpath", 0o755)
# 2) CL.Exe launcher (intercepts VC71_CL_WSL from vc71_verify.py)
rxdk_bin = "/mnt/c/Program Files (x86)/RXDK/xbox/bin/vc71"
os.makedirs(rxdk_bin, exist_ok=True)
with open(os.path.join(rxdk_bin, "CL.Exe"), "w") as f:
    f.write('#!/bin/bash\nexport WINEARCH=win32\nexport WINEDEBUG=-all\ncl="C:\\Program Files\\Microsoft Visual C++ Toolkit 2003\\bin\\cl.exe"\nexec wine "$cl" "$@"\n')
os.chmod(os.path.join(rxdk_bin, "CL.Exe"), 0o755)
# 3) RXDK include symlink
rxdk_inc = os.path.expanduser("~/.wine/drive_c/Program Files (x86)/RXDK/xbox/include")
vc_inc = os.path.expanduser("~/.wine/drive_c/Program Files/Microsoft Visual C++ Toolkit 2003/include")
os.makedirs(os.path.dirname(rxdk_inc), exist_ok=True)
if not os.path.exists(rxdk_inc):
    os.symlink(vc_inc, rxdk_inc)
print("  ✓ Shims configured: /usr/local/bin/wslpath -> winepath, /mnt/c/.../CL.Exe -> wine cl.exe.")
# --- 6. GENERATE DECL.H ---
print("\n>>> [6/6] Generating build/generated/decl.h from kb.json...")
os.chdir(halo_dir)
os.makedirs("build/generated", exist_ok=True)
res_decl = subprocess.run(["python3", "tools/analysis/knowledge.py", "--gen-header", "build/generated/decl.h"], capture_output=True, text=True)
if res_decl.returncode == 0:
    print(f"  ✓ decl.h generated successfully ({os.path.getsize('build/generated/decl.h'):,} bytes)")
else:
    print("  ⚠ Note on decl.h generation:", res_decl.stderr.strip()[:200])
print("\n" + "=" * 80)
print("🎉 ENVIRONMENT READY FOR VERIFICATION & BYTEMATCH SEARCH ON mainrevised2!")
print("=" * 80)


  HALO CE XBOX (BUILD 2276) - MSVC 7.1 & BYTEMATCH ENVIRONMENT SETUP
  Branch: mainrevised2 | Target: cachebeta.xbe

>>> [1/6] Staging Assets from Google Drive (if available)...
Mounted at /content/drive
  Staging VCToolkitSetup.exe from /content/drive/MyDrive/MSVC 7.1...
  ✓ Asset ready: /content/VCToolkitSetup.exe (32,952,488 bytes)
  Staging cachebeta.xbe from /content/drive/MyDrive/MSVC 7.1...
  ✓ Asset ready: /content/cachebeta.xbe (3,395,584 bytes)

>>> [2/6] Synchronizing repository (branch: mainrevised2)...
  Cloning https://github.com/nickarcade/halo.git on branch 'mainrevised2'...
  ✓ cachebeta.xbe verified in halo-patched/ (MD5: c7869590a1c64ad034e49a5ee0c02465) [MATCH]

>>> [3/6] Installing Linux packages (Wine32, LLVM, libclang, Python analysis tools)...


KeyboardInterrupt: 

### 🧪 Step 2: Smoke Test — MSVC 7.1 Single Function Verification
Test that `vc71_verify.py` compiles and scores a function cleanly using the transparent Wine toolchain.


In [ ]:
# @title 🔍 Run Single Function Verification Smoke Test
target_tu = "src/halo/interface/event_manager.c" # @param {type:"string"}
target_func = "FUN_000dc000" # @param {type:"string"}

import subprocess, os

os.chdir("/content/halo")
cmd = ["python3", "tools/verify/vc71_verify.py", target_tu]
if target_func:
    cmd.extend(["--function", target_func])
cmd.append("--no-cache")

print(f"Executing: {' '.join(cmd)}\n")
res = subprocess.run(cmd, capture_output=True, text=True)
print(res.stdout)
if res.stderr:
    print("STDERR:", res.stderr)


---
## 🔬 Step 3: Structural Audit Evidence (`raw_xbe_structural.py`)
`bytematch.py` relies on audit records under `artifacts/raw_xbe_structural/` to categorize instruction differences (`operand_order`, `register`, `stack_offset`, etc.) and measure aligned byte accuracy.


In [ ]:
# @title 📊 Populate Raw-XBE Structural Audit Records
# @markdown Runs structural audit across ported functions to populate `artifacts/raw_xbe_structural/`.
workers = 4 # @param {type:"integer"}
specific_source = "src/halo/interface/event_manager.c" # @param {type:"string"}
import subprocess, os
os.chdir("/content/halo")
cmd = ["python3", "tools/verify/raw_xbe_structural.py", "populate", "--workers", str(workers)]
if specific_source.strip():
    cmd.extend(["--source", specific_source.strip()])
cmd_str = " ".join(cmd)
print(f"Running: {cmd_str}\n")
res = subprocess.run(cmd, capture_output=True, text=True)
print(res.stdout)
if res.stderr:
    print(res.stderr)


---
## 🎯 Step 4: Byte-Match Queue & Candidate Selection
Inspect the top near-exact functions ranked by fewest non-knock-on differences.


In [ ]:
# @title 📋 Inspect Candidate Queue for Bytematch
queue_limit = 30 # @param {type:"integer"}
min_accuracy = 0.85 # @param {type:"number"}

import subprocess, os

os.chdir("/content/halo")
cmd = ["python3", "tools/bytematch/bytematch.py", "queue", "--limit", str(queue_limit), "--min-accuracy", str(min_accuracy)]
res = subprocess.run(cmd, capture_output=True, text=True)
print(res.stdout)
if res.stderr:
    print(res.stderr)


---
## ⚡ Step 5: AST-Guided Byte-Match Search (`bytematch.py search`)
This searches for semantics-preserving source rewrites via `libclang` (commutative swaps, relational flips, `if/else` inversions), compiles each variant with VC7.1 under Wine, and records improvements in `artifacts/bytematch/ledger.jsonl`.


In [ ]:
# @title 🔍 Search AST Rewrites for a Function
target_function = "event_manager_tab_process" # @param {type:"string"}
apply_patch = False # @param {type:"boolean"}

import subprocess, os

os.chdir("/content/halo")
if not target_function.strip():
    print("Please provide a target function name (e.g. from the queue above).")
else:
    cmd = ["python3", "tools/bytematch/bytematch.py", "search", target_function.strip()]
    if apply_patch:
        cmd.append("--apply")
    cmd_str = " ".join(cmd)
    print(f"Running: {cmd_str}\n")
    res = subprocess.run(cmd, capture_output=True, text=True)
    print(res.stdout)
    if res.stderr:
        print(res.stderr)


### 🚀 Batch Search Across Top Candidates
Automatically runs `bytematch.py` across the highest-priority queue targets.


In [ ]:
# @title ⚡ Run Batch Byte-Match Search
batch_limit = 5 # @param {type:"integer"}
batch_min_accuracy = 0.80 # @param {type:"number"}
import subprocess, os
os.chdir("/content/halo")
cmd = [
    "python3", "tools/bytematch/bytematch.py", "batch",
    "--limit", str(batch_limit),
    "--min-accuracy", str(batch_min_accuracy)
]
cmd_str = " ".join(cmd)
print(f"Running: {cmd_str}\n")
res = subprocess.run(cmd, capture_output=True, text=True)
print(res.stdout)
if res.stderr:
    print(res.stderr)


### 📈 Ledger Rules & Success Rates
Summarizes the ledger (`artifacts/bytematch/ledger.jsonl`): per `(difference class, transform)` success rates.


In [ ]:
# @title 📊 View Bytematch Rules & Measured Success Rates
import subprocess, os

os.chdir("/content/halo")
res = subprocess.run(["python3", "tools/bytematch/bytematch.py", "rules"], capture_output=True, text=True)
print(res.stdout)
if res.stderr:
    print(res.stderr)


---
## 🏆 Step 6: PAL Campaign Tracking & Metrics
Track raw-byte completion metrics and residual classifications.


In [ ]:
# @title 📊 PAL Campaign Metrics & Residuals
import subprocess, os

os.chdir("/content/halo")
res = subprocess.run(["python3", "tools/bytematch/pal_campaign.py", "metrics"], capture_output=True, text=True)
print(res.stdout)
if res.stderr:
    print(res.stderr)


---
## ✅ Step 7: Required Gates & Regression Check
Whenever source changes are applied, run the mandatory verification gates:
1. `check_lift_hazards.py --changed-only`
2. `vc71_verify.py` on the modified translation unit


In [ ]:
# @title 🛡️ Run Hazard Gate & Git Diff
import subprocess, os

os.chdir("/content/halo")
print("=== Git Status ===")
subprocess.run(["git", "status", "--short"], text=True)

print("\n=== Hazard Scan on Modified Sources ===")
res_h = subprocess.run(["python3", "tools/audit/check_lift_hazards.py", "--changed-only"], capture_output=True, text=True)
print(res_h.stdout)
if res_h.stderr:
    print(res_h.stderr)
